
$\qquad$ $\qquad$$\qquad$  **TDA206/DIT206 Discrete Optimization: Home Assignment 2 -- Integer LP and Relaxation** <br />
$\qquad$ $\qquad$$\qquad$                   **Grader: Marc Constantin** <br />
$\qquad$ $\qquad$$\qquad$                   **Due Date: 17th Feb** <br />
$\qquad$ $\qquad$$\qquad$                   **Submitted by: Elias Söderberg, 20030507-8792., elisod@chalmers.se** <br />
$\qquad$ $\qquad$$\qquad$                   **Submitted by: Jonathan Sunnerstam, 20020420-6734., jonsun@chalmers.se** <br />


---


General guidelines:
*   All solutions to theoretical and pratical problems must be submitted in this ipynb notebook, and equations wherever required, should be formatted using LaTeX math-mode.
*   All discussion regarding practical problems, along with solutions and plots should be specified in this notebook. All plots/results should be visible such that the notebook do not have to be run. But the code in the notebook should reproduce the plots/results if we choose to do so.
*   Your name, personal number and email address should be specified above.
*   All tables and other additional information should be included in this notebook.
*   Before submitting, make sure that your code can run on another computer. That all plots can show on another computer including all your writing. It is good to check if your code can run here: https://colab.research.google.com.


# Question 1.
(5 points) 
There are 4 space colonies, each of which  requires a certain number of plasma conduits. There are 3 starbases in the vicinity. Each of them has total number of conduits they can spare and supply to the colonies. For each pair of starbase and colony, there is an associated cost for sending a cargo ship  (each of which carries one plasma conduit), as shown in the table below:


\begin{array}{l|c|c|c|c|c} 
      & Triacus & New Berlin  & Strnad  & Vega  & supply\\ \hline
 Farpoint &   6 &  9 & 10 & 8 & 35\\
 Yorktown &  9 & 5 & 16 & 14 & 40\\
 Earhart & 12 &  7 & 13 & 9 & 50\\ \hline
    demand & 20 &30&30&45& \left(\sum=125\right) \\ 
\end{array}

Your goal is to supply the colonies the plasma conduits they need, at minimum cost.


Formulate a LP to solve the problem, explain why the solution is integral with a proof.

$$
\begin{aligned}
&\sum_{j=1}^{4}x_{ij} \geq \text{demand}_i \quad& \text{For all rows} \ i \\
&\sum_{i=1}^{3}x_{ij} \leq \text{supply}_j \quad& \text{For all columns} \ i \\
\end{aligned}
$$

In [27]:
%pip install -q cvxpy
%pip install -q numpy
import cvxpy as cp
import numpy as np

costs = np.array([
    [6,  9, 10,  8], # Farpoint
    [9,  5, 16, 14], # Yorktown
    [12, 7, 13,  9]  # Earhart
])

supply = np.array([35, 40, 50])
demand = np.array([20, 30, 30, 45])

# x[i, j]: amount shipped base i -> col j
x = cp.Variable((3, 4), nonneg=True)

# sum(rows) == supply, sum(cols) == demand
constraints = [
    cp.sum(x, axis=1) <= supply,
    cp.sum(x, axis=0) >= demand,
]

obj = cp.Minimize(cp.sum(cp.multiply(costs, x)))
prob = cp.Problem(obj, constraints)
prob.solve()

print(f"Optimal cost: {np.round(prob.value, 2)}")
bases = ["Farpoint", "Yorktown", "Earhart"]
colonies = ["Triacus", "NewBerlin", "Strnad", "Vega"]
rows, cols = x.value.shape
for i in range(rows):
    for j in range(cols):
        amount = x.value[i, j]
        print(f"{np.round(amount, 2)} conduits from {bases[i]} to {colonies[j]}")

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Optimal cost: 1020.0
10.0 conduits from Farpoint to Triacus
0.0 conduits from Farpoint to NewBerlin
25.0 conduits from Farpoint to Strnad
0.0 conduits from Farpoint to Vega
10.0 conduits from Yorktown to Triacus
30.0 conduits from Yorktown to NewBerlin
0.0 conduits from Yorktown to Strnad
0.0 conduits from Yorktown to Vega
0.0 conduits from Earhart to Triacus
0.0 conduits from Earhart to NewBerlin
5.0 conduits from Earhart to Strnad
45.0 conduits from Earhart to Vega


# Discussion Part 1
As the problem hinted, the solution is integral. This is because of the lemmas presented in lecture 4 (Hoffman-Kruskal), which allows one that the solution to an LP problem is integral, iff the constraint coefficient matrix $A$ in $Ax = b$ is totally unimodular (TU) (and $b$ consists only of integers). In this problem, the matrix $A$ is a matrix of size $7 \times 12$, where the rows represent starbases and colonies, and every column corresponds to a variable $x_{ij}$ where $x_{ij}$ represents the number of plasma conduits shipped from starbase $i$ to colony $j$. Written out, the matrix looks like (with a line separating starbases and colonies):

$$
A = \begin{bmatrix}
1 & 1 & 1 & 1 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 1 & 1 & 1 & 1 & 0 & 0 & 0 & 0 \\
0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 1 & 1 & 1 & 1 \\
\hline
1 & 0 & 0 & 0 & 1 & 0 & 0 & 0 & 1 & 0 & 0 & 0 \\
0 & 1 & 0 & 0 & 0 & 1 & 0 & 0 & 0 & 1 & 0 & 0 \\
0 & 0 & 1 & 0 & 0 & 0 & 1 & 0 & 0 & 0 & 1 & 0 \\
0 & 0 & 0 & 1 & 0 & 0 & 0 & 1 & 0 & 0 & 0 & 1 
\end{bmatrix}
$$

The first $3$ rows ensure that the sum of shipments from each starbase equals its supply capacity, while the latter $4$ rows ensure that the sum of shipments to each colony meets its demand.

As presented in lecture 4 Theorem 5, sufficient conditions for TU are that $A$ only contains entries $\in {0, 1, -1}$; each column has at most two non-zeros; the matrix is bipartite. As easily seen, this holds for $A$, and thus the optimal solution will always be integral.


# Question 2.

Recall the Minimum Weight Vertex Cover (VC) Problem: Given an undirected graph $G=(V, E)$, with node set $V$ and edge set $E$, where each node has a positive weight $w(v)$ associated with it (see the attached figure), the goal is to select a subset $V'\subseteq V$ of nodes such that every edge has at least one node incident to it, and the total selected node weight $\sum_{v\in V'} w(v)$ is minimized. 

* (4 points) Formulate the ILP for the VC problem for the attached graph, and solve it using **CVXPY** integer solver, for instance, `myVar = cp.Variable(<dim>, integer=True)`.
* (2 points) Pass to the LP relaxation and solve it using **CVXPY** and comment on the relation between the two solutions.
* (2 points) Apply the rounding rule discussed in class to the optimal LP solution to obtain a solution to the ILP and compare it to the optimal ILP solution.

<!-- ![title](vertex_cover_example.png) -->


In [23]:
%pip install -q cvxpy
%pip install -q numpy
import cvxpy as cp
import numpy as np

# nodes[i] = cost of node number i
# edges[i] = (i, j): edge between nodes[i] and nodes[j]
def vc_solver(nodes: list[int], edges: list[tuple[int, int]], ilp: bool = True):
    if ilp:
        node_vars = cp.Variable(len(nodes), boolean=True)
    else:
        node_vars = cp.Variable(len(nodes), nonneg=True)
    obj = cp.Minimize(
        cp.sum(cp.multiply(nodes, node_vars))
    )

    constraints = []
    for (i, j) in edges:
        constraints.append(node_vars[i] + node_vars[j] >= 1)

    prob = cp.Problem(obj, constraints)
    prob.solve()
    if ilp:
        picks = [i + 1 for i, node in enumerate(node_vars) if node.value != 0]
        print(f"Select nodes: {picks} with combined cost: {prob.value}")
    else:
        print("Values for nodes:")
        for i, node in enumerate(node_vars, start=1):
            print(f"Node {i}: {np.round(node.value, 2)}")
        print(f"with combined cost: {np.round(prob.value, 2)}")

    return node_vars
            
nodes = [1, 3, 4, 2, 2, 4]
edges = [
    (0, 1),
    (1, 2), (1, 4),
    (2, 3), (2, 4),
    (3, 4),
    (4, 5)
]

print("Solving the problem using ILP:")
vc_solver(nodes, edges)
print("LP relaxation of the problem:")
nodes = vc_solver(nodes, edges, ilp=False)

rounded_values = [1 if node >= 1/2 else 0 for node in nodes.value]
print("Rounded values:")
for i, val in enumerate(rounded_values):
    print(f"Node {i + 1} has value: {val}")

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Solving the problem using ILP:
Select nodes: [2, 4, 5] with combined cost: 7.0
LP relaxation of the problem:
Values for nodes:
Node 1: 0.48
Node 2: 0.52
Node 3: 0.48
Node 4: 0.52
Node 5: 1.0
Node 6: 0.0
with combined cost: 7.0
Rounded values:
Node 1 has value: 0
Node 2 has value: 1
Node 3 has value: 0
Node 4 has value: 1
Node 5 has value: 1
Node 6 has value: 0


# Discussion part 2 and 3
The two solutions yield the same final cost. This means that the ILP solution is an optimal solution as it is a feasable solution to the relaxation (which optimizes among every possible feasable solution). For the LP relaxation, applying the rounding rule $S_{LP} = \{v \in V : x_v^* \geq \frac{1}{2}\}$ from chapter $3.3$ in the book yields the same solution as the ILP version of the problem.

# Question 3.

Consider a number of interpreters (Olof, Petra, Qamar,
  Rachel, Soren and Tao), as well as a set of languages (Arab,
  Bengali, Cantonese, Dutch, English, French and German). Each
  interpreter speaks a number of different languages (abbreviated by
  first letter), and has a certain per-diem integer cost:

\begin{array}{lll}
Interpreter & Languages & Cost\\
O & ABD & 3\\
P & C & 1\\
Q & CDG & 1\\
R & B & 2\\
S & G & 4\\
T & EF & 1\\
\end{array}

* (2 points) A *hypergraph* is a structure $H = (V,E)$ where $V$ is a set of vertices and $E$ is a collection of subsets of $V$. The special case when all subsets $e \in E$ have size exactly $2$ corresponds to the familiar case of a graph. A vertex cover in such a hypergraph is a subset $U \subseteq V$ such that $e \cap U \not = \emptyset$ for each $e \in E$ (note that this reduces to the usual vertex cover in graphs). Show that the problem of finding interpreters can be formulated as a vertex cover problem in a sutable hypergraph.
* (4 points) Develop a ILP formulation to finding the vertex cover of minimum cost in a hypergraph. The hypergraph can be represented as a $|V| \times |E|$ binary matrix $A$ where $A[i,j] = 1$ iff vertex $i$ is in edge $j$ and 0 otherwise. The costs for vertices are in an array $\texttt{c}$ where the cost of picking vertex $i$ is $c[i]$. Use the ILP formulation for the VC problem to hire the cheapest set of interpreters such that all languages are covered. Input the data above manuallly and solve it using **CVXPY**'s integer solver.
* (2 points) Pass to the LP relaxation and solve it using **CVXPY**.
* (2 points) Explain why the two solutions above are same (different).


# As a Vertex Cover Problem
The problem can be formulated as a vertex cover problem ina suitable hypergrah $H = (V, E)$ by letting $V$ represent the set of interpreters, and having $E$ be the set of hyperedges representing languages. Each hyperedge is defined by the subset of vertices that can speak that language.
$$
\begin{aligned}
&V = \{O, P, Q, R, S, T\} \\
&E = \{e_a, e_b, e_c, e_d, e_e, e_f, e_g\}
\end{aligned}
$$
where 
$$
\begin{aligned}
&e_a = \{O\} \\
&e_b = \{O, R\} \\
&e_c = \{P, Q\} \\
&e_d = \{O, Q\} \\
&e_e = \{T\} \\
&e_f = \{T\} \\
&e_g = \{Q, S\} \\
\end{aligned}
$$
Thus, finding a vertex cover equates to finding a set of interpreters covering all languages.

# ILP Formulation
Let $x_i$ be a binary decision variable for each interpreter $i \in V$, where $x_i = 1$ if interpreter $i$ is chosen, $0$ otherwise. Let $c_i$ denote the cost of interpreter $i$. The objective function becomes: 
$$
\text{Minimize} \sum_{i \in V} c_i x_i
$$

and the constraint that every language must be covered by at least one chosen interpreter:

$$
\sum_{i \in e_j} x_i \geq 1 \quad \forall \ \text{hyperedges} \ j \in E
$$

The problem suggests the hypergraph representation as a binary matrix $A = |V| \times |E|$, but using the transpose $A^T$ adheres better to standard constraint notation $Ax \geq b$, allowing for a clean implementation below.

In [19]:
#implementation
%pip install -q cvxpy
%pip install -q numpy
import cvxpy as cp
import numpy as np

def solve(ilp=True):

    interpreters = ["O", "P", "Q", "R", "S", "T"]
    costs = np.array([3, 1, 1, 2, 4, 1])
    if ilp:
        x = cp.Variable(len(interpreters), boolean=True)
    else:
        x = cp.Variable(len(interpreters))    
    
    A = np.array([
        #O  P  Q  R  S  T
        [1, 0, 0, 0, 0, 0], # A
        [1, 0, 0, 1, 0, 0], # B
        [0, 1, 1, 0, 0, 0], # C
        [1, 0, 1, 0, 0, 0], # D
        [0, 0, 0, 0, 0, 1], # E
        [0, 0, 0, 0, 0, 1], # F
        [0, 0, 1, 0, 1, 0], # G
    ])
    # @: dot product
    obj = cp.Minimize(costs @ x)
    if ilp:
        constraints = [A @ x >= 1]
    else:
        constraints = [
            A @ x >= 1,
            x >= 0,
            x <= 1
        ]
    
    prob = cp.Problem(obj, constraints)
    prob.solve()

    print(f"Final cost: {np.round(prob.value, 2)}")
    print("Chosen interpreters")
    for i, val in enumerate(x.value):
        if val != 0: 
            print(f"{interpreters[i]}: {np.round(val, 2)} (cost {costs[i]})")

print("ILP solution:")
solve()
print("\nLP relaxation")
solve(ilp=False)
    

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
ILP solution:
Final cost: 5.0
Chosen interpreters
O: 1.0 (cost 3)
Q: 1.0 (cost 1)
T: 1.0 (cost 1)

LP relaxation
Final cost: 5.0
Chosen interpreters
O: 1.0 (cost 3)
P: 0.0 (cost 1)
Q: 1.0 (cost 1)
R: 0.0 (cost 2)
S: -0.0 (cost 4)
T: 1.0 (cost 1)


# Discussion: Same Answers
As shown, both the ILP and LP relaxation yields the same answers. Looking at the problem, it is not difficult to see why. Both the interpreters "Olof" and "Tao" are forced picks, as they are the only interpreters covering Arabic and English, respectively. Other than Arabic and English, they together also cover Bengali, Dutch, and French. This leaves Cantonese and German left to cover. The cheapest way to cover both is by the use of Qamar, which is integral. Therefore, the LP relaxation is naturally integral.

# Question 4. 
Consider the ILP and its LP relaxation corresponding to the VC problem for the graph $G$ given in the data file. This is a ***random graph*** $G(n,p)$ with $n=100$ vertices generated as follows: for each pair of vertices **independently**, we add an edge with probability $p=0.1$ (so the graph has about 1000 edges).

* **a**. (2 points) Find the optimal solution using **CVXPY**'s integer solver.
* **b**. (2 points) Solve the LP relaxation using **CVXPY** and apply the rounding rule discussed in class to obtain a vertex cover. Compare it to the optimal solution in part (a).
* **c**. (6 points) Consider the following rounding rule: we build up the vertex cover incrementally starting with $S:= \emptyset$. Now consider the edges in $G$ in any order. If an edge $(u,v)$ is already covered by a vertex in $S$, do nothing. Otherwise add to $S$ the vertex $u$ if $x^*(u) \geq x^*(v)$, or $v$ otherwise (where ${\bf x}^*$ is the LP optimum solution computed in part (b).  Comment why this also results in a vertex cover and has cost no more than that corresponding to the rounding rule in part (b). Compare the cost of the solution produced by this rule to the optimal solution.

In [45]:
%pip install -q cvxpy
%pip install -q numpy

import numpy as np
import cvxpy as cp

# nodes[i] = cost of node number i
# edges[i] = (i, j): edge between nodes[i] and nodes[j]
def vc_solver(nodes: list[int], edges: list[tuple[int, int]], ilp: bool = True):
    if ilp:
        node_vars = cp.Variable(len(nodes), boolean=True)
    else:
        node_vars = cp.Variable(len(nodes), nonneg=True)
    obj = cp.Minimize(nodes @ node_vars)

    constraints = []
    for (i, j) in edges:
        constraints.append(node_vars[i] + node_vars[j] >= 1)

    prob = cp.Problem(obj, constraints)
    prob.solve()
    if ilp:
        picks = [i + 1 for i, node in enumerate(node_vars) if node.value > 0.01]
        print(f"Select nodes: {picks} with combined cost: {prob.value}")
    else:
        print("Values for nodes:")
        for i, node in enumerate(node_vars, start=1):
            print(f"Node {i}: {np.round(node.value, 2)}")
        print(f"with combined cost: {np.round(prob.value, 2)}")

    return node_vars

nodes = []
edges = []
with open("graph.txt") as f:
    data = f.read()
    data = data.splitlines()
    for n1, row in enumerate(data):
        nodes.append(1) # cost
        for n2, char in enumerate("".join(row.split())):
            if char == "1" and n1 < n2: edges.append((n1, n2))
            

    
print("Solving the problem using ILP:")
vc_solver(nodes, edges)
print("LP relaxation of the problem:")
nodes = vc_solver(nodes, edges, ilp=False)

rounded_values = [1 if node >= 0.5 else 0 for node in nodes.value]
print("Rounded values:")
print([i + 1 for i, value in enumerate(rounded_values) if value == 1])
print(f"Optimal cost of rounding: {sum(rounded_values)}")

# part 3:
S = set()
for (n1, n2) in edges:
    if (n1 in S) or (n2 in S): continue
    to_add = n1 if nodes.value[n1] >= nodes.value[n2] else n2
    S.add(to_add)
print(S)
print(f"Cost of Part C rounding: {len(S)}")

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Solving the problem using ILP:
Select nodes: [1, 2, 3, 5, 6, 7, 8, 11, 13, 14, 17, 18, 20, 21, 23, 25, 26, 28, 29, 30, 31, 32, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 45, 46, 47, 48, 49, 50, 53, 54, 55, 56, 57, 59, 60, 63, 64, 66, 68, 69, 71, 73, 74, 76, 78, 79, 80, 81, 82, 83, 85, 86, 88, 90, 91, 93, 94, 96, 97, 98] with combined cost: 70.0
LP relaxation of the problem:
Values for nodes:
Node 1: 0.5
Node 2: 0.5
Node 3: 0.5
Node 4: 0.5
Node 5: 0.5
Node 6: 0.5
Node 7: 0.5
Node 8: 0.5
Node 9: 0.5
Node 10: 0.5
Node 11: 0.5
Node 12: 0.5
Node 13: 0.5
Node 14: 0.5
Node 15: 0.5
Node 16: 0.5
Node 17: 0.5
Node 18: 0.5
Node 19: 0.5
Node 20: 0.5
Node 21: 0.5
Node 22: 0.5
Node 23: 0.5
Node 24: 0.5
Node 25: 0.5
Node 26: 0.5
Node 27: 0.5
Node 28: 0.5
Node 29: 0.5
Node 30: 0.5
Node 31: 0.5
Node 32: 0.5
Node 33: 0.5
Node 34: 0.5
Node 35: 0.5
Node 36: 0.5
Node 37: 0.5
Node

# Discussion part b and c
The optimal value in part b is better than the solution given by the CVXPY ILP solver. This could be because of limitations to the CVXPY ILP solver.

The values given by the LP relaxation suggests that the random graph (where edges are randomly distributed) is best solved by partially including every node. This could be an error in the implementation, but no errors are found by us. It is indeed a vertex cover, as the rounding algorithm iterates through every edge and ensures that each edge is covered; If an edge is not covered, then one node will always be added to cover it. The statement that the optimal cost should be no more than the cost produced by the rounding rule in part b hints that our implementation is wrong.